# Environment Testing — Gymnasium TradingEnv

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

from stable_baselines3.common.env_checker import check_env
from src.environment.trading_env import TradingEnv
from src.agents import buy_and_hold, ma_crossover
from src.evaluation.metrics import compute_all
from src.evaluation.backtester import compare_strategies

# Load config
with open("../configs/dqn_config.yaml") as f:
    cfg = yaml.safe_load(f)

ENV_CFG = cfg["environment"]
print("Config loaded. ENV_CFG:", ENV_CFG)

# Load normalised data (environment input)
df_train_norm = pd.read_csv("../data/normalized/train.csv", index_col=0, parse_dates=True)
df_val_norm   = pd.read_csv("../data/normalized/val.csv",   index_col=0, parse_dates=True)
df_test_norm  = pd.read_csv("../data/normalized/test.csv",  index_col=0, parse_dates=True)

# Load raw feature data (for baselines — human-readable prices)
df_train = pd.read_csv("../data/features/train.csv", index_col=0, parse_dates=True)
df_val   = pd.read_csv("../data/features/val.csv",   index_col=0, parse_dates=True)
df_test  = pd.read_csv("../data/features/test.csv",  index_col=0, parse_dates=True)

print(f"Train: {len(df_train_norm)} rows  |  Val: {len(df_val_norm)}  |  Test: {len(df_test_norm)}")

## 1. SB3 Environment Compliance Check

In [ ]:
env = TradingEnv(df_train_norm, ENV_CFG)
check_env(env, warn=True)
print(f"\ncheck_env: PASSED")
print(f"Observation space : {env.observation_space}")
print(f"Action space      : {env.action_space}")
print(f"Episode length    : {env.n_steps} steps")

## 2. Initial Observation & Observation Space Bounds

In [ ]:
obs, _ = env.reset()
OBS_LABELS = ["open", "high", "low", "close", "volume",
               "sma_10", "sma_50", "rsi_norm", "momentum_norm",
               "position", "portfolio_pct"]
print("Initial observation vector:")
for label, val in zip(OBS_LABELS, obs):
    print(f"  {label:<16s}: {val:.4f}")

# Verify all obs stay in [0, 1] across a random episode
env2 = TradingEnv(df_train_norm, ENV_CFG)
obs, _ = env2.reset()
all_obs = [obs]
done = False
while not done:
    obs, _, terminated, truncated, _ = env2.step(env2.action_space.sample())
    done = terminated or truncated
    all_obs.append(obs)

arr = np.array(all_obs)
print(f"\nObs min across episode : {arr.min():.6f}")
print(f"Obs max across episode : {arr.max():.6f}")
print(f"All obs in [0, 1]      : {bool((arr >= 0).all() and (arr <= 1).all())}")

## 3. Scripted Episode: Buy-Hold-Sell sequence

In [ ]:
# Scripted run: Buy on step 0, hold, then Sell on the last step
env_script = TradingEnv(df_train_norm, ENV_CFG)
obs, _ = env_script.reset()

records = []
for step in range(len(df_train_norm)):
    if step == 0:
        action = 1  # Buy
    elif step == len(df_train_norm) - 1:
        action = 2  # Sell at the end
    else:
        action = 0  # Hold

    obs, reward, terminated, truncated, info = env_script.step(action)
    info["reward"] = reward
    info["date"] = df_train_norm.index[step]
    records.append(info)
    if terminated or truncated:
        break

script_df = pd.DataFrame(records).set_index("date")
print(f"Episode return: {env_script.total_return:.2f}%")
print(script_df[["action", "price", "portfolio_value", "position", "reward"]].to_string())

## 4. Portfolio Value Comparison: B&H vs MA Crossover (training set)

In [ ]:
import os
os.makedirs("../results/figures", exist_ok=True)

cfg_baseline = dict(initial_balance=ENV_CFG["initial_balance"],
                    transaction_cost=ENV_CFG["transaction_cost"])

bnh_train = buy_and_hold.run(df_train, **cfg_baseline)
mac_train  = ma_crossover.run(df_train, **cfg_baseline)

# ----- Chart -----
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

ax1, ax2 = axes
dates = df_train.index

# Top: portfolio value
ax1.plot(dates, bnh_train["portfolio_value"], label="Buy-and-Hold", linewidth=2)
ax1.plot(dates, mac_train["portfolio_value"],  label="MA Crossover", linewidth=2, linestyle="--")
ax1.axhline(ENV_CFG["initial_balance"], color="grey", linewidth=1, linestyle=":", label="Initial balance")
ax1.set_ylabel("Portfolio Value (USDT)")
ax1.set_title("Baseline Strategy Performance — Training Set (Sep–Dec 2024)")
ax1.legend()
ax1.grid(alpha=0.3)

# Mark MA crossover buy/sell events
signals = ma_crossover.generate_signals(df_train)
buys  = df_train.index[signals == 1]
sells = df_train.index[signals == 2]
for d in buys:
    ax1.axvline(d, color="green", alpha=0.5, linewidth=1.5, linestyle="--")
for d in sells:
    ax1.axvline(d, color="red",   alpha=0.5, linewidth=1.5, linestyle="--")

# Bottom: MA crossover position (shaded regions)
position = mac_train["position"]
for i in range(len(position)):
    if position.iloc[i] == 1:
        ax2.axvspan(dates[i], dates[min(i+1, len(dates)-1)],
                    alpha=0.3, color="green")
ax2.set_ylabel("Position (MA)")
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["Cash", "Long"])
ax2.set_xlabel("Date")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/figures/03_baseline_comparison_train.png", dpi=150, bbox_inches="tight")
plt.show()

# Metrics table
m_bnh = compute_all(bnh_train["portfolio_value"])
m_mac = compute_all(mac_train["portfolio_value"])
table = compare_strategies({"Buy-and-Hold": m_bnh, "MA-Crossover": m_mac})
print("\nMetrics — Training Set")
print(table.to_string())

## 5. Random-Policy Episode — Distribution of Rewards

In [ ]:
# Run N random episodes to understand the reward distribution the DQN must learn from
N_EPISODES = 500
random_returns = []
env_rand = TradingEnv(df_train_norm, ENV_CFG)

for _ in range(N_EPISODES):
    obs, _ = env_rand.reset()
    done = False
    while not done:
        obs, _, terminated, truncated, _ = env_rand.step(env_rand.action_space.sample())
        done = terminated or truncated
    random_returns.append(env_rand.total_return)

random_returns = np.array(random_returns)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(random_returns, bins=40, edgecolor="white", linewidth=0.5)
ax.axvline(random_returns.mean(), color="red",    linewidth=2, label=f"Mean: {random_returns.mean():.1f}%")
ax.axvline(np.median(random_returns), color="orange", linewidth=2, linestyle="--",
           label=f"Median: {np.median(random_returns):.1f}%")
bnh_return = (bnh_train["portfolio_value"].iloc[-1] / ENV_CFG["initial_balance"] - 1) * 100
ax.axvline(bnh_return, color="green", linewidth=2, linestyle=":",
           label=f"Buy-and-Hold: {bnh_return:.1f}%")
ax.set_xlabel("Episode Total Return (%)")
ax.set_ylabel("Count")
ax.set_title(f"Return Distribution — {N_EPISODES} Random Episodes (Training Set)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/03_random_policy_returns.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Random policy — mean return : {random_returns.mean():.2f}%")
print(f"Random policy — std         : {random_returns.std():.2f}%")
print(f"Random policy — min/max     : {random_returns.min():.2f}% / {random_returns.max():.2f}%")
print(f"Buy-and-Hold  — return      : {bnh_return:.2f}%")